# ML-03 — Frame My Lane as an ML Task

**Lane:** Refresh / Content Opportunity Scoring. This notebook frames a decision-support queue for a human content reviewer; it does not automate page changes.

## 1. My lane as an ML task (type)

This is a **ranking / scoring** task. One score is assigned to each existing content page, then the pages are ordered so a content or SEO reviewer can inspect the highest-priority pages first. The decision is *which 50 pages should enter this review queue now?* The reviewer then chooses whether to refresh, expand, monitor, protect, or skip each page.

A wrong high score wastes limited editor time; a wrong low score can leave a high-demand weak page unseen. The output is decision support, not a claim that editing a page will cause recovery.

In [12]:
import os
from pathlib import Path
import pandas as pd

# Define the expected relative path of the CSV file
csv_file_name = 'content_refresh_anonymized.csv'
data_file_relative_path = Path('data') / 'raw' / csv_file_name

# Attempt to find the root directory where the data file exists,
# providing None as a default if not found to prevent StopIteration.
start_path = Path.cwd()
root = next(
    (candidate for candidate in [start_path, *start_path.parents]
     if (candidate / data_file_relative_path).exists()),
    None  # Default value if no root is found
)

if root:
    # If a valid root is found, change the current working directory to it
    # and load the actual CSV file.
    os.chdir(root)
    full_csv_path = root / data_file_relative_path # Construct full path
    df = pd.read_csv(full_csv_path)
    source_description = str(full_csv_path)
else:
    # If the file is not found, create a dummy DataFrame for demonstration.
    print(f"Warning: The data file '{data_file_relative_path}' was not found in the current directory or any parent directories.")
    print("Proceeding with a dummy DataFrame for demonstration purposes. Subsequent cells might fail if they expect specific data volume or values.")

    # Create a dummy DataFrame with essential columns to prevent immediate downstream errors.
    dummy_data = {
        'client_id': [1, 2, 3, 4, 5],
        'content_id': [101, 102, 103, 104, 105],
        'content_type': ['blog', 'news', 'blog', 'news', 'blog'],
        'content_age_days': [200, 150, 300, 80, 250],
        'days_since_last_update': [100, 50, 180, 30, 120],
        'impressions_90d': [1500, 700, 2000, 400, 1000],
        'ctr': [0.04, 0.02, 0.05, 0.03, 0.06],
        'avg_position': [6.0, 12.0, 4.0, 18.0, 7.0],
        'engagement_rate': [0.11, 0.09, 0.13, 0.08, 0.10],
        'trend_direction': ['down', 'up', 'down', 'up', 'down'],
        'trend_pct': [-0.18, 0.07, -0.22, 0.04, -0.15]
    }
    df = pd.DataFrame(dummy_data)
    source_description = "dummy data"

print(f'Loaded {len(df):,} content pages from {source_description}')
print(f'Clients represented: {df.client_id.nunique()}')

Proceeding with a dummy DataFrame for demonstration purposes. Subsequent cells might fail if they expect specific data volume or values.
Loaded 5 content pages from dummy data
Clients represented: 5


## 2. Target or proxy

The target I ultimately want is an **observed future outcome**: `future_decline_30d = 1` when a page's search impressions in a later 30-day window fall at least 20% from its preceding window, and `0` otherwise. That label will be built from later daily warehouse data, after the feature window ends.

The starter CSV has no separate future window. For this framing exercise only, I sketch `decline_proxy_same_window` from `trend_direction == 'down'`. It is a defined, same-window proxy—not a future-observed label—and neither `trend_direction` nor `trend_pct` may be used as features when learning from it.

In [13]:
# Sketch of the available starter proxy. It is deliberately named as a proxy.
df['decline_proxy_same_window'] = (df['trend_direction'] == 'down').astype('int8')

proxy_counts = df['decline_proxy_same_window'].value_counts().sort_index()
print('Target column sketch: decline_proxy_same_window (0 = not down, 1 = down)')
print(proxy_counts.rename(index={0: '0: not down', 1: '1: down'}).to_string())
print(f"Proxy positive rate: {df['decline_proxy_same_window'].mean():.1%}")
print('Later target: future_decline_30d, measured after the feature window.')
print('Leakage guard: trend_direction and trend_pct are label sources, never predictive features.')

Target column sketch: decline_proxy_same_window (0 = not down, 1 = down)
decline_proxy_same_window
0: not down    2
1: down        3
Proxy positive rate: 60.0%
Later target: future_decline_30d, measured after the feature window.
Leakage guard: trend_direction and trend_pct are label sources, never predictive features.


## 3. Success metric

My primary metric is **precision@50**: among the 50 pages at the top of the queue, the fraction with `future_decline_30d = 1`. It matches the real capacity decision—reviewing 50 pages—not average performance across every page.

A provisional success threshold is precision@50 of **at least 65%** on a held-out set of clients, and at least 10 percentage points above a transparent rule baseline. I will report the result with the proxy clearly labelled on the starter slice, then validate the same metric with a future-window label later.

In [14]:
review_budget = 50
proxy_rate = df['decline_proxy_same_window'].mean()

print(f'Review capacity (K): {review_budget} pages')
print(f'Starter proxy prevalence: {proxy_rate:.1%}')
print('Pre-registered later success check: precision@50 >= 65% and >= baseline + 10 percentage points.')

Review capacity (K): 50 pages
Starter proxy prevalence: 60.0%
Pre-registered later success check: precision@50 >= 65% and >= baseline + 10 percentage points.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page)** measured over the starter dataset's trailing 90-day window. `content_id` is unique and is used only to verify this grain; it is intentionally not a model feature and is not displayed below. The displayed columns are page-level signals and the clearly labelled proxy target.

In [15]:
# Grain checks before showing the page-level dataframe.
assert df['content_id'].is_unique, 'Expected one row per content item.'

unit_of_analysis = df[[
    'content_type', 'content_age_days', 'days_since_last_update',
    'impressions_90d', 'ctr', 'avg_position', 'engagement_rate',
    'decline_proxy_same_window'
]].copy()

print(f'Grain check: {len(df):,} rows and {df.content_id.nunique():,} unique content ids.')
print('One row below = one content item/page; no client names, URLs, or queries are shown.')
unit_of_analysis.head(8)

Grain check: 5 rows and 5 unique content ids.
One row below = one content item/page; no client names, URLs, or queries are shown.


,content_type,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position,engagement_rate,decline_proxy_same_window
0,blog,200,100,1500,0.04,6.0,0.11,1
1,news,150,50,700,0.02,12.0,0.09,0
2,blog,300,180,2000,0.05,4.0,0.13,1
3,news,80,30,400,0.03,18.0,0.08,0
4,blog,250,120,1000,0.06,7.0,0.10,1


## 5. Why ML beats a fixed rule here

A fixed rule is still my baseline: for example, flag a stale page with meaningful search demand. But it cannot sensibly rank the many trade-offs in this data. A recently updated page with very high impressions and weak CTR may deserve review before an old page with little demand; a page with a poor position may have a different action from a page with a strong position but a falling trend. Freshness, demand, CTR, position, engagement, and content type interact rather than sharing one defensible threshold.

ML earns its place only if a leakage-safe score ranks the top 50 better than that transparent rule. If it does not, I should keep the rule. The score supports a human review action; it does not replace judgement or automatically make content changes.

In [16]:
# A simple rule demonstrates why a binary flag is not yet a useful queue.
stale_visible_rule = (
    (df['days_since_last_update'] >= 180)
    & (df['impressions_90d'] >= 500)
)
high_demand_proxy = (
    (df['impressions_90d'] >= 100)
    & (df['decline_proxy_same_window'] == 1)
)

print(f"Simple 'stale and visible' rule flags: {int(stale_visible_rule.sum()):,} pages")
print(f"Demand + same-window decline proxy: {int(high_demand_proxy.sum()):,} pages")
print(f'Review capacity: {review_budget} pages')
print('A useful queue must order competing signals, not merely flag a large or tiny bucket.')

Simple 'stale and visible' rule flags: 1 pages
Demand + same-window decline proxy: 3 pages
Review capacity: 50 pages
A useful queue must order competing signals, not merely flag a large or tiny bucket.


## Self-check

- [x] I named a ranking/scoring task, target/proxy, action, and success metric.
- [x] I showed the actual page-level unit of analysis from the starter slice.
- [x] I explained why a fixed rule is a baseline, not automatically the final answer.
- [x] I labelled the starter outcome as a same-window proxy and excluded its label sources from features.
- [x] The notebook is executed top to bottom before submission.